# Exp1A: D6 Adjacent Sentence-Pair Swap

A more naturalistic disruption than D4: keeps sentences intact,
only disrupts local discourse order by swapping adjacent sentence pairs.

Original: S1 S2 S3 S4 S5 → D6: S2 S1 S4 S3 S5

Same tokens, same sentence-internal word order, same multiset.
Expected: intact ≈ D0 > D6 > D4

Pilot: wiki_zh + wiki_ja, Llama, 10 shuffles.

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import json, math, os, gc, random, time, re
from pathlib import Path
from collections import Counter
from scipy import stats
from scipy.ndimage import uniform_filter1d
from tqdm.auto import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from google.colab import drive
drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/LRTIA/Results/Exp1A_D6')
BASE.mkdir(parents=True, exist_ok=True)
# Also load existing D4/intact results for comparison
FORMAL_BASE = Path('/content/drive/MyDrive/LRTIA/Results/Exp1A_formal')
DATA = Path('/content/drive/MyDrive/LRTIA/Data')

CORPORA = {
    'wiki_zh': DATA / 'wiki_multilingual/zh_articles.jsonl',
    'wiki_ja': DATA / 'wiki_multilingual/ja_articles.jsonl',
}

MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B'
C = 100
TARGET_LEN = 30
TARGET_FRACS = [0.25, 0.50, 0.75]
MIN_BEFORE = C + 10
N_SHUFFLES = 10
SEED = 20260429

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print('Setup done')

In [ ]:
# === Load model ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16),
    device_map='auto'
)
model.eval()
print('Llama loaded')

In [ ]:
# === D6: Adjacent sentence-pair swap ===

# Sentence boundary characters
SENT_BOUNDARIES = set('. ? ! 。 ？ ！ ؟'.split())

def segment_into_sentences(token_ids, tokenizer):
    """Segment token sequence into sentence-like units using punctuation.
    
    Returns list of (start_idx, end_idx) spans within token_ids.
    Each span is a sentence-like unit. Fragments at boundaries are kept.
    """
    units = []
    current_start = 0
    
    for i, tok_id in enumerate(token_ids):
        tok_str = tokenizer.decode([tok_id]).strip()
        # Check if this token ends with a sentence boundary
        is_boundary = False
        for ch in tok_str:
            if ch in SENT_BOUNDARIES:
                is_boundary = True
                break
        
        if is_boundary:
            units.append((current_start, i + 1))
            current_start = i + 1
    
    # Final fragment
    if current_start < len(token_ids):
        units.append((current_start, len(token_ids)))
    
    return units


def d6_adjacent_sentence_swap(context_tokens, tokenizer):
    """Swap adjacent sentence pairs in the context.
    
    S1 S2 S3 S4 S5 → S2 S1 S4 S3 S5
    
    Returns (disrupted_tokens, n_units, eligible) where:
    - disrupted_tokens: list of token IDs after swap
    - n_units: number of sentence-like units found
    - eligible: True if >= 4 units (otherwise swap is trivial)
    """
    units = segment_into_sentences(context_tokens, tokenizer)
    n_units = len(units)
    
    if n_units < 4:
        return None, n_units, False
    
    # Swap adjacent pairs from oldest to newest
    swapped_units = []
    i = 0
    while i < n_units - 1:
        # Swap unit i and i+1
        swapped_units.append(units[i + 1])
        swapped_units.append(units[i])
        i += 2
    # If odd number, last unit stays
    if i < n_units:
        swapped_units.append(units[i])
    
    # Flatten back to tokens
    result = []
    for start, end in swapped_units:
        result.extend(context_tokens[start:end])
    
    return result, n_units, True


# === Test D6 on a sample ===
test_doc = None
with open(CORPORA['wiki_zh']) as f:
    test_doc = json.loads(f.readline())

full_ids = tokenizer.encode(test_doc['text'], add_special_tokens=False)
ts = int(len(full_ids) * 0.5)
ctx = full_ids[ts-C:ts]

d6_ctx, n_units, eligible = d6_adjacent_sentence_swap(ctx, tokenizer)

print(f'Test: {n_units} sentence units, eligible: {eligible}')
if d6_ctx:
    # Assertions
    assert len(d6_ctx) == len(ctx), f'Length mismatch: {len(d6_ctx)} vs {len(ctx)}'
    assert Counter(d6_ctx) == Counter(ctx), 'Token multiset changed!'
    # No target leakage
    target = full_ids[ts:ts+TARGET_LEN]
    assert not any(t in d6_ctx for t in target if ctx.count(t) == 0), 'Target leakage!'
    print(f'  Length: {len(d6_ctx)} (same as original: {len(d6_ctx) == len(ctx)})')
    print(f'  Multiset preserved: {Counter(d6_ctx) == Counter(ctx)}')
    
    # Show the swap
    units = segment_into_sentences(ctx, tokenizer)
    print(f'\n  Original unit boundaries: {[(s,e) for s,e in units]}')
    print(f'  Unit lengths: {[e-s for s,e in units]}')
    for i, (s, e) in enumerate(units[:6]):
        text = tokenizer.decode(ctx[s:e])[:60]
        print(f'    S{i}: "{text}..."')
print('D6 function verified')

In [ ]:
# === PPL + corrected marginal functions ===

@torch.no_grad()
def compute_ppl_nll(context_tokens, target_tokens):
    if len(target_tokens) < 2:
        return float('inf'), float('inf')
    full = list(context_tokens) + list(target_tokens)
    ts = len(context_tokens)
    ids = torch.tensor([full], device=model.device)
    out = model(ids)
    logits = out.logits[0]
    nll_sum = 0.0
    cnt = 0
    for i in range(ts, len(full) - 1):
        lp = torch.log_softmax(logits[i], dim=-1)
        nll_sum += -lp[full[i+1]].item()
        cnt += 1
    del out, logits
    torch.cuda.empty_cache()
    if cnt == 0: return float('inf'), float('inf')
    mn = nll_sum / cnt
    return math.exp(mn), mn

def compute_curves(cond_ctx, target_tokens):
    max_c = len(cond_ctx)
    o_ppl, o_nll, s_ppl, s_nll = [], [], [], []
    for c in range(max_c + 1):
        prefix = cond_ctx[-c:] if c > 0 else []
        ppl, nll = compute_ppl_nll(prefix, target_tokens)
        o_ppl.append(ppl); o_nll.append(nll)
        if c == 0:
            s_ppl.append(ppl); s_nll.append(nll)
        else:
            rng = random.Random(SEED + c)
            sp_l, sn_l = [], []
            for _ in range(N_SHUFFLES):
                shuf = list(prefix)
                rng.shuffle(shuf)
                sp, sn = compute_ppl_nll(shuf, target_tokens)
                if not math.isinf(sp): sp_l.append(sp); sn_l.append(sn)
            s_ppl.append(np.mean(sp_l) if sp_l else ppl)
            s_nll.append(np.mean(sn_l) if sn_l else nll)
    dists = list(range(1, max_c + 1))
    mo = [o_ppl[d-1] - o_ppl[d] for d in dists]
    ms = [s_ppl[d-1] - s_ppl[d] for d in dists]
    delta = [a - b for a, b in zip(mo, ms)]
    return {
        'distances': dists,
        'ordered_ppl': o_ppl, 'shuffled_ppl': s_ppl,
        'm_ordered': mo, 'm_shuffled': ms, 'delta_ppl': delta,
    }

print(f'Functions ready — {N_SHUFFLES} shuffles per c')

In [ ]:
# === Run D6 + intact on pilot corpora ===

d6_stats = {}  # corpus -> {eligible_rate, mean_units, ...}

for corpus_name, corpus_path in CORPORA.items():
    print(f'\n{"="*60}')
    print(f'{corpus_name}')
    print(f'{"="*60}')
    
    docs = []
    with open(corpus_path) as f:
        for line in f:
            docs.append(json.loads(line))
    
    for cond_name in ['intact', 'D6']:
        cache = BASE / f'llama_{corpus_name}_{cond_name}.json'
        if cache.exists():
            with open(cache) as f: n = len(json.load(f))
            print(f'  {cond_name}: cached ({n})'); continue
        
        t0 = time.time()
        results = []
        n_eligible = 0
        n_total = 0
        all_n_units = []
        
        for doc in tqdm(docs, desc=f'{corpus_name}/{cond_name}'):
            full_ids = tokenizer.encode(doc['text'], add_special_tokens=False)
            n = len(full_ids)
            
            for frac in TARGET_FRACS:
                ts = int(n * frac)
                te = min(ts + TARGET_LEN, n)
                if ts < MIN_BEFORE or te - ts < 5: continue
                
                ctx = full_ids[ts-C:ts]
                target = full_ids[ts:te]
                n_total += 1
                
                if cond_name == 'intact':
                    cond_ctx = list(ctx)
                elif cond_name == 'D6':
                    cond_ctx, n_units, eligible = d6_adjacent_sentence_swap(ctx, tokenizer)
                    all_n_units.append(n_units)
                    if not eligible:
                        continue  # skip ineligible
                    n_eligible += 1
                    # Assertions
                    assert len(cond_ctx) == C, f'Length {len(cond_ctx)} != {C}'
                    assert Counter(cond_ctx) == Counter(ctx), 'Multiset changed'
                
                r = compute_curves(cond_ctx, target)
                r['doc_id'] = doc.get('doc_id', '')
                r['target_frac'] = frac
                if cond_name == 'D6':
                    r['n_units'] = n_units
                results.append(r)
        
        with open(cache, 'w') as f:
            json.dump(results, f)
        elapsed = time.time() - t0
        print(f'  {cond_name}: {len(results)} results in {elapsed/60:.1f} min')
        
        if cond_name == 'D6':
            elig_rate = n_eligible / n_total if n_total > 0 else 0
            mean_units = np.mean(all_n_units) if all_n_units else 0
            d6_stats[corpus_name] = {
                'eligible_rate': elig_rate,
                'mean_units': mean_units,
                'n_eligible': n_eligible,
                'n_total': n_total,
            }
            print(f'    Eligibility: {n_eligible}/{n_total} ({elig_rate:.0%})')
            print(f'    Mean sentence units: {mean_units:.1f}')
        
        if results:
            md = np.mean([np.mean(r['delta_ppl']) for r in results])
            print(f'    Mean Δ(ppl): {md:.6f}')

In [ ]:
# === Results table ===

print(f'\n{"="*70}')
print('D6 RESULTS: intact vs D6 vs D4')
print(f'{"="*70}')

print(f'\n{"Corpus":<12} {"Cond":<10} {"TotalΔ":>10} {"NearΔ":>10} {"MidΔ":>10} {"FarΔ":>10}')
print(f'{"":>12} {"":>10} {"d=1..100":>10} {"d=1..30":>10} {"d=31..70":>10} {"d=71..100":>10}')
print('-' * 65)

for corpus_name in CORPORA:
    for cond_name, cond_base in [('intact', BASE), ('D6', BASE), ('D4', FORMAL_BASE)]:
        if cond_name == 'D4':
            cp = cond_base / f'llama_{corpus_name}_D4.json'
        else:
            cp = cond_base / f'llama_{corpus_name}_{cond_name}.json'
        if not cp.exists():
            print(f'{corpus_name:<12} {cond_name:<10} {"—":>10}')
            continue
        with open(cp) as f: results = json.load(f)
        if not results: continue
        
        all_delta = np.array([r['delta_ppl'] for r in results])
        mean_curve = np.mean(all_delta, axis=0)
        
        total = np.mean(mean_curve)
        near = np.mean(mean_curve[:30])
        mid = np.mean(mean_curve[30:70])
        far = np.mean(mean_curve[70:])
        
        print(f'{corpus_name:<12} {cond_name:<10} {total:>10.4f} {near:>10.4f} {mid:>10.4f} {far:>10.4f}')
    print()

# D6 eligibility stats
print('D6 Eligibility:')
for corpus_name, st in d6_stats.items():
    print(f'  {corpus_name}: {st["n_eligible"]}/{st["n_total"]} ({st["eligible_rate"]:.0%}), '
          f'mean {st["mean_units"]:.1f} sentence units')

In [ ]:
# === Visualization ===
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(CORPORA), figsize=(7*len(CORPORA), 5))
if len(CORPORA) == 1: axes = [axes]

colors = {'intact': 'blue', 'D6': 'orange', 'D4': 'green'}

for idx, corpus_name in enumerate(CORPORA):
    ax = axes[idx]
    
    for cond_name, cond_base in [('intact', BASE), ('D6', BASE), ('D4', FORMAL_BASE)]:
        if cond_name == 'D4':
            cp = cond_base / f'llama_{corpus_name}_D4.json'
        else:
            cp = cond_base / f'llama_{corpus_name}_{cond_name}.json'
        if not cp.exists(): continue
        with open(cp) as f: results = json.load(f)
        if not results: continue
        curve = np.mean([r['delta_ppl'] for r in results], axis=0)
        smooth = uniform_filter1d(curve, 5)
        ax.plot(range(1, len(smooth)+1), smooth, color=colors[cond_name],
                linewidth=2, label=cond_name)
    
    ax.axhline(0, color='gray', linestyle=':', alpha=0.3)
    ax.set_title(corpus_name, fontweight='bold')
    ax.set_xlabel('Distance d')
    ax.set_ylabel('Corrected Marginal Δ')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.15)

plt.suptitle('D6 (sentence-pair swap) vs Intact vs D4 (full reverse)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE / 'fig_D6_pilot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved')